In [4]:
# 代码作用：原糖样品分析明细（色值、干燥失重转置）v3 - 数值字段填充字符串"0"
# 2026/03/24
# 数据治理工程师：yushumeng

import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col, lit, when

spark = SparkSession.builder \
    .appName("raw_sugar_sample_analysis") \
    .enableHiveSupport() \
    .getOrCreate()

# ========================
# 1. 彻底清理旧表（元数据 + HDFS 数据目录）
# ========================
target_table = "app.app_lims_raw_sugar_sample_analysis_detailed_f_1h"
hdfs_table_path = "hdfs://vpc-minibaselas/user/hive/warehouse/app.db/app_lims_raw_sugar_sample_analysis_detailed_f_1h"

spark.sql(f"DROP TABLE IF EXISTS {target_table}")
print(f"已删除 Hive 表: {target_table}")

try:
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    fs = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
    path = spark.sparkContext._jvm.org.apache.hadoop.fs.Path(hdfs_table_path)
    if fs.exists(path):
        fs.delete(path, True)
        print(f"已删除 HDFS 目录: {hdfs_table_path}")
    else:
        print(f"HDFS 目录不存在，无需删除: {hdfs_table_path}")
except Exception as e:
    print(f"删除 HDFS 目录失败（可能无权限或路径错误）: {e}")

# ========================
# 2. 数据转置处理
# ========================
analysis_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        sam_sample_original_no,
        test_name1,
        test_test_value AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_productioncategory = '榨蔗生产'
        AND ord_sample_name LIKE '%原糖%'
        AND test_name1 IN ('色值', '干燥失重')
)
SELECT 
    company,
    crushing_season,
    record_date,
    work_shift,
    ord_sample_name,
    sam_sample_original_no,
    MAX(CASE WHEN test_name1 = '色值' THEN test_value END) AS color_value_str,
    MAX(CASE WHEN test_name1 = '干燥失重' THEN test_value END) AS loss_on_drying_str
FROM raw_data
GROUP BY company, crushing_season, record_date, work_shift, ord_sample_name, sam_sample_original_no
ORDER BY record_date, work_shift, ord_sample_name, sam_sample_original_no
"""

print("执行转置查询...")
df = spark.sql(analysis_sql)

# 清洗数值字段：合法数字保留原字符串，非法值（含空字符串）填充字符串 "0"
def clean_numeric_fill_zero_str(col_name):
    return when(col(col_name).rlike('^[0-9.]+$'), col(col_name)) \
           .otherwise(lit("0")).alias(col_name)

df = df.withColumn("color_value", clean_numeric_fill_zero_str("color_value_str")) \
       .withColumn("loss_on_drying", clean_numeric_fill_zero_str("loss_on_drying_str")) \
       .drop("color_value_str", "loss_on_drying_str")

# 添加时间戳和来源标识（全部转为字符串）
df = df.withColumn("dw_update_time", current_timestamp().cast("string")) \
       .withColumn("source_flg", lit("LIMS"))

# 将所有字段统一转为字符串（确保类型一致）
df = df.select([col(c).cast("string") for c in df.columns])

# 按最终列顺序选择（与建表语句一致）
df = df.select(
    "company", "crushing_season", "record_date", "work_shift",
    "ord_sample_name", "sam_sample_original_no",
    "color_value", "loss_on_drying",
    "dw_update_time", "source_flg"
)

# ========================
# 3. 显式创建新表（全 STRING）
# ========================
create_sql = f"""
CREATE TABLE {target_table} (
    company STRING COMMENT '公司编码',
    crushing_season STRING COMMENT '榨季',
    record_date STRING COMMENT '采样日期',
    work_shift STRING COMMENT '班次',
    ord_sample_name STRING COMMENT '样品名称',
    sam_sample_original_no STRING COMMENT '样品原始编号',
    color_value STRING COMMENT '色值（IU）',
    loss_on_drying STRING COMMENT '干燥失重（%）',
    dw_update_time STRING COMMENT '数据加载时间戳',
    source_flg STRING COMMENT '数据来源标识'
)
COMMENT '原糖样品分析明细表（色值、干燥失重转置）'
STORED AS PARQUET
TBLPROPERTIES ('parquet.compression'='SNAPPY')
"""
spark.sql(create_sql)
print(f"已创建新表: {target_table}")

# ========================
# 4. 写入数据
# ========================
print("写入数据...")
df.write.mode("overwrite").insertInto(target_table)

print("数据写入完成！")
spark.stop()

/opt/tiger/spark/python/pyspark/context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


已删除 Hive 表: app.app_lims_raw_sugar_sample_analysis_detailed_f_1h
HDFS 目录不存在，无需删除: hdfs://vpc-minibaselas/user/hive/warehouse/app.db/app_lims_raw_sugar_sample_analysis_detailed_f_1h
执行转置查询...
已创建新表: app.app_lims_raw_sugar_sample_analysis_detailed_f_1h
写入数据...
数据写入完成！
